In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_small/checkpoints/post_cell_20_try_3.pickle

In [ ]:
%%cudf.pandas.profile
### cell 21 ###

### cell 21 optimized ###

# define the question and (possibly blank) x-axis title
question_of_interest_cell33 = 'For how many years have you been writing code and/or programming?'
title_for_x_axis_cell33 = ''

# count, normalize and round a Series
def count_then_return_percent_33(series):
    counts = series.value_counts(dropna=False)
    return (counts * 100 / counts.sum()).round(1)

# combine percentages for multiple years into one DataFrame
def combine_subset_of_data_from_multiple_years_33(question_of_interest, x_axis_title, include_2017=False):
    year_sources = [
        ('2018', responses_df_2018_cell10, 'How long have you been writing code to analyze data?'),
        ('2019', responses_df_2019_cell10, 'How long have you been writing code to analyze data (at work or at school)?'),
        ('2020', responses_df_2020, None),
        ('2021', responses_df_2021, None),
        ('2022', responses_df_2022_cell10, None),
    ]
    if include_2017:
        year_sources.insert(0, ('2017', responses_df_2017, None))

    replace_maps = {
        '2018': {
            '< 1 year': '< 1 years',
            'I have never written code but I want to learn': 'I have never written code',
            'I have never written code and I do not want to learn': 'I have never written code',
            '1-2 years': '1-3 years',
            '20-30 years': '20+ years',
            '30-40 years': '20+ years',
            '40+ years': '20+ years',
        },
        '2019': {'1-2 years': '1-3 years'},
        '2020': {'1-2 years': '1-3 years'},
    }

    frames = []
    for year, df_src, orig_col in year_sources:
        # rename the column if it differs from our canonical question
        df = df_src.rename(columns={orig_col: question_of_interest}) if orig_col else df_src
        s = df[question_of_interest]
        # apply any year-specific recoding
        if year in replace_maps:
            s = s.replace(replace_maps[year])
        # compute percentages
        pct = count_then_return_percent_33(s)
        # build a DataFrame, reset index so categories become a column
        df_year = pct.sort_index().reset_index()
        # name the category column using x_axis_title (can be blank string)
        df_year.columns = [x_axis_title, 'percentage']
        df_year['year'] = year
        frames.append(df_year)

    # stack all years together
    df_combined = pd.concat(frames, ignore_index=True)
    # ensure column order matches [x_axis_title, 'percentage', 'year']
    return df_combined[[x_axis_title, 'percentage', 'year']]

# run and inspect
programming_df_combined = (
    combine_subset_of_data_from_multiple_years_33(
        question_of_interest_cell33,
        title_for_x_axis_cell33
    )
    .sort_values(['year', 'percentage'])
)
programming_df_combined.info()